In [1]:
from IPython.display import Markdown, display, Latex
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import os

proj = ccrs.epsg(25830)
buffer = 1000

# nombre_municipio = 'San Martín de Unx'

# municipios = gpd.read_file('../../navarra/BRUTOS/MUNICIPIOS/')
# municipio = municipios[municipios['MUNICIPIO'] == nombre_municipio]
# cod_mun = municipio.CMUNICIPIO.values[0]
# bbox = municipio.geometry.iloc[0].bounds
# minx, miny, maxx, maxy = bbox

# aspect = (maxx - minx) / (maxy -miny)
# h = 20
# v = round(h / aspect)

In [3]:
import rasterio

# MDT, SLOPE, ASPECT

In [2]:
from pathlib import Path
import rasterio
from rasterio.merge import merge

# 1. Gather all input raster files
input_dir = Path("BRUTOS/MDT/")
raster_paths = list(input_dir.glob("MDT*.tif"))

# 2. Open each raster file in read mode
src_files_to_mosaic = []
for path in raster_paths:
    src = rasterio.open(path)
    src_files_to_mosaic.append(src)

# 3. Merge the datasets into an array and get the new spatial transform
merged_array, out_transform = merge(src_files_to_mosaic)

# 4. Copy the metadata from one of the original datasets
out_meta = src_files_to_mosaic[0].meta.copy()

# 5. Update metadata with the new dimensions and spatial transformation
out_meta.update({
    "driver": "GTiff",
    "height": merged_array.shape[1],
    "width": merged_array.shape[2],
    "transform": out_transform
})

# 6. Save the merged mosaic to a new file
output_path = "BRUTOS/merged_mdt.tif"
with rasterio.open(output_path, "w", **out_meta) as dest:
    dest.write(merged_array)

# 7. Always close your source files to free up system resources
for src in src_files_to_mosaic:
    src.close()

In [9]:
!gdal_translate -of AAIGrid BRUTOS/merged_mdt.tif BRUTOS/MDT/mdt.asc

Input file size is 1113, 766
0...10...20...30...40...50...60...70...80...90...100 - done.


In [5]:
!gdaldem slope BRUTOS/merged_mdt.tif BRUTOS/slope.tif -p

# !gdal_translate -of AAIGrid BRUTOS/MDT/slope.tif BRUTOS/MDT/slope.asc

0...10...20...30...40...50...60...70...80...90...100 - done.


In [4]:
!gdaldem aspect BRUTOS/merged_mdt.tif BRUTOS/aspect.tif
# !gdal_translate -of AAIGrid BRUTOS/MDT/aspect.tif BRUTOS/MDT/aspect.asc

0...10...20...30...40...50...60...70...80...90...100 - done.


# MODELOS DE COMBUSTIBLE

In [25]:
import rasterio 
from shapely. geometry import box

with rasterio.open('BRUTOS/merged_mdt.tif') as dataset:
    bbox = dataset.bounds

rect = box(*list(bbox))
print(rect)

POLYGON ((648984 4706299, 648984 4725449, 621159 4725449, 621159 4706299, 648984 4706299))


In [35]:
mc = gpd.read_file('BRUTOS/MC/FOREST_Pol_ModeCombus.shp')
mc = mc[mc.intersects(rect) == True]
mc.geometry = mc.intersection(rect)

In [56]:
mc.loc[~mc.IDCOMBUSTI.isin([str(x) for x in range(1,10)]),'IDCOMBUSTI'] = '0'

In [57]:
mc.IDCOMBUSTI.unique()

array(['1', '2', '3', '4', '5', '6', '7', '8', '9', '0'], dtype=object)

In [65]:
import rasterio
from rasterio.features import rasterize
import shapely.geometry

geoms_values = [(x.geometry, int(x.IDCOMBUSTI)) for i, x in mc.iterrows()]
# print(geoms_values)

# 3. Rasterize the geometries
rasterized_array = rasterize(
    shapes=geoms_values,
    out_shape=merged_array.shape[1:3],
    fill=0,
    transform=out_transform,
    all_touched=False, # Set to True to burn in all touched pixels
    dtype='uint8'
)

# # 4. (Optional) Write to a new GeoTIFF file
with rasterio.open(
    'BRUTOS/MC/mc_raster.tif',
    'w',
    **out_meta
) as dst:
    dst.write(rasterized_array, 1)

In [63]:
merged_array.shape

(1, 766, 1113)